# 逻辑回归：解决二分类问题


- 分类模型
> 1. 回归——>计算分数（连续值的预测）
> 2. 对分数进行 `sigmoid`，得到一个概率——`sigmoid` 的值域$(0,1)$
> 3. 分类——>根据分数和阈值判断分类结果

- 执行过程
> 1. 训练
> 2. 初始化 w，b
> 3. 梯度下降
> 4. 预测

In [2]:
import torch
import math

def logistic_regression_predict(
    x_train: torch.Tensor,
    y_train: torch.Tensor,
    x_test: torch.Tensor,
    learning_rate: float,
    num_steps: int,
) -> torch.Tensor:

    n, d = x_train.shape[0], x_train.shape[1]

    bias = torch.zeros(1, dtype = x_train.dtype) # n * 1
    weights = torch.zeros(d, dtype = x_train.dtype) # d * 1
    def my_sigmoid(x):
        return 1.0 / (1.0 + torch.exp(-x))

    def predict(x, _weights, _bias):
        results = x @ _weights + _bias # n * 1
        # results.sigmoid()
        return my_sigmoid(results) # n * 1
    def gradient_weight(predicts, x, y):
        n = x.shape[0]
        return (x.T @(predicts - y)) / n
    def gradient_bias(predicts, y):
        n = y.shape[0]
        return (predicts - y).mean()

    def gradient_descent(_grd_w, _grd_b, w, b, lr):
        w = w - lr * _grd_w
        b = b - lr * _grd_b
        return w, b

    for _ in range(num_steps) :
        y_predicts = predict(x_train, weights, bias)
        grd_w = gradient_weight(y_predicts, x_train, y_train)
        grd_b = gradient_bias(y_predicts, y_train)

        weights, bias = gradient_descent(grd_w, grd_b, weights,bias, learning_rate)


    predicts = predict(x_test, weights, bias)
    return predicts

In [5]:
x_train = torch.tensor([[-2],[-1],[1],[2]], dtype = torch.float32)
y_train = torch.tensor([0,0,1,1],dtype=torch.float32)
x_test = torch.tensor([[-1.5],[0],[1.5]],dtype=torch.float32)
learning_rate = 0.2
num_steps = 20

# 预期结果[0.11403120309114456,0.5,0.885968804359436]
logistic_regression_predict(x_train, y_train, x_test, learning_rate, num_steps)

tensor([0.1140, 0.5000, 0.8860])

# 利用逻辑回归识别涂鸦

## 读取图片

In [1]:
import numpy as np
import torch
from PIL import Image

APPLE_PATH = "datas/LR_Fruits/apples.png"
BANANA_PATH = "datas/LR_Fruits/bananas.png"

torch.set_num_threads(1)

def load_doodles(path):
    # 大图由 100 行 15 列、共 1500 张 28×28 的涂鸦拼成，先把它切回一张张涂鸦
    grid = np.array(Image.open(path))
    print(grid.shape)
    return grid.reshape(100, 28, 15, 28).swapaxes(1, 2).reshape(1500, 28, 28)
# 1500张图片，每张大小 28*28
apples = load_doodles(APPLE_PATH)
print(apples.shape)
bananas = load_doodles(BANANA_PATH)
print(bananas.shape)

(2800, 420)
(1500, 28, 28)
(2800, 420)
(1500, 28, 28)


## 预处理数据


- 处理图片数据
> 1. 拼接
> 2. 特征缩放——这里使用归一化
> 3. 变成 tensor 格式

In [2]:
# 将两张图片拼接起来
images_concatenate = np.concatenate([apples, bananas], axis = 0)
print(images_concatenate.shape)

# 归一化，这里的最大值其实就是255，最小值0
mins = images_concatenate.min(axis=(1, 2), keepdims=True)
maxs = images_concatenate.max(axis=(1, 2), keepdims=True)
# 归一化 + 反相
images_normalized = 1 - (images_concatenate - mins) / (maxs - mins)

# 变成 tensor
images = torch.tensor(images_normalized, dtype=torch.float32)
print(images.shape)

# 将图片展开成一维
input = images.reshape(images.shape[0], images.shape[1] * images.shape[2])
print(input.shape)

(3000, 28, 28)
torch.Size([3000, 28, 28])
torch.Size([3000, 784])


- 标签数据

In [3]:
labels=torch.cat([torch.ones(apples.shape[0]), torch.zeros(bananas.shape[0])])
print(labels.shape)
print(labels)
print(labels.sum())

torch.Size([3000])
tensor([1., 1., 1.,  ..., 0., 0., 0.])
tensor(1500.)


## 开始训练

- 打乱数据，并分成 6：2：2的比例

In [4]:
# 生成一个 0～2999 的随机序列，打乱数据
order = torch.randperm(3000, generator=torch.Generator().manual_seed(42))

train_ids = order[:1800]
test_id = order[1800: 2400]
val_ids = order[2400:]

print(train_ids.size(), test_id.size(), val_ids.size())
x_train, y_train = input[train_ids], labels[train_ids]
x_test, y_test = input[test_id], labels[test_id]
x_val, y_val = input[val_ids], labels[val_ids]

print(x_train.shape, y_train.shape, x_test.shape, y_test.shape, x_val.shape, y_val.shape)

torch.Size([1800]) torch.Size([600]) torch.Size([600])
torch.Size([1800, 784]) torch.Size([1800]) torch.Size([600, 784]) torch.Size([600]) torch.Size([600, 784]) torch.Size([600])


- 初始化
> 1. `weight` 和 `bias`
> 2. 损失函数
> 3. 优化器

In [5]:
weights = torch.zeros(784, requires_grad=True)
bias = torch.zeros(1, requires_grad=True)
loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD([weights, bias], lr=0.2)

- 开始训练
> 1. 计算误差
> 2. 计算梯度
> 3. 梯度下降
> 4. 不停迭代

In [8]:
num_steps=500
train_losses, val_losses = [], []
for step in range(num_steps):
    # 计算结果
    logits = x_train @ weights + bias

    # 误差
    train_loss = loss_fn(logits, y_train)
    with torch.no_grad():
        val_logits = x_val @ weights + bias
        val_loss = loss_fn(val_logits, y_val)

    train_losses.append(train_loss.item())
    val_losses.append(val_loss.item())

    if step % 100 == 0:
        print(f"训练集误差{train_loss}, 验证集误差{val_loss}")

    # 梯度下降
    optimizer.zero_grad()# 清空梯度
    train_loss.backward()# 计算梯度
    optimizer.step()# 梯度下降



训练集误差0.6931472420692444, 验证集误差0.6931473016738892
训练集误差0.10740480571985245, 验证集误差0.12239141017198563
训练集误差0.09232541918754578, 验证集误差0.10951927304267883
训练集误差0.0846463069319725, 验证集误差0.10527661442756653
训练集误差0.0792500451207161, 验证集误差0.10352947562932968


- 检测结果
> 1. 预测
> 2. 统计预测结果

In [10]:
def accuracy(x, y):
    with torch.no_grad():
        probabilities = torch.sigmoid(x @ weights + bias)
    return ((probabilities >= 0.5).float() == y).float().mean().item()


def accuracy_2(x, y):
    weights_2 = torch.zeros(784, requires_grad=True)
    bias_2 = torch.zeros(1, requires_grad=True)
    with torch.no_grad():
        probabilities = torch.sigmoid(x @ weights_2 + bias_2)
    return ((probabilities >= 0.5).float() == y).float().mean().item()
print("训练前")
print(f"训练集准确率：{accuracy_2(x_train, y_train):.1%}")
print(f"验证集准确率：{accuracy_2(x_val, y_val):.1%}")
print(f"测试集准确率：{accuracy_2(x_test, y_test):.1%}")
print('-'*75)
print('训练后')
print(f"训练集准确率：{accuracy(x_train, y_train):.1%}")
print(f"验证集准确率：{accuracy(x_val, y_val):.1%}")
print(f"测试集准确率：{accuracy(x_test, y_test):.1%}")

训练前
训练集准确率：49.8%
验证集准确率：50.8%
测试集准确率：49.8%
---------------------------------------------------------------------------
训练后
训练集准确率：97.8%
验证集准确率：95.7%
测试集准确率：95.2%
